## How to use this app

**Press `Run all` in the toolbar above, then wait for the screen to appear under the second cell.** The first cell installs the tool and connects your Drive; the second one draws the screen. Running them one at a time works too, but `Run all` is the shortest path and keeps them in the right order.

Then, on the screen:

1. **Choose CSV** - pick the price file from your computer. It does not need to be in Drive.
2. **Login** - paste your VTEX login value into the password field.
3. **Check this file** - nothing is written by this step. The tool reads the current price of every SKU and region in your file and reports what it found.
4. **Read the verdict.** Three numbers: what will be written, what needs your check, what is blocked and excluded. Blocked rows are dropped from the upload; the rest of the file still goes.
5. **Review the warnings**, then press the single confirm button to unlock the upload.
6. **Upload** - the button states exactly what one click does, including how many existing prices it removes.
7. **Read the read-back.** After the upload the tool waits for VTEX pricing to settle and then reads every row back. Only *matched* counts as success.

**To undo the last upload:** press *Put the previous prices back*. It clears the screen, states what it would restore and for how many pairs, and writes nothing until you confirm.

**Prefer a dry run?** Tick *Check only - do not upload* before pressing *Check this file*.


In [ ]:
# Cell 1 - Setup. Run this first.
!pip install -q --upgrade "git+https://github.com/Hmart-Ecommerce/vtex-fixed-price-uploader.git" ipywidgets
from google.colab import drive
drive.mount('/content/drive')

import os

# Name the folder once; let the notebook find it.
#
# The same folder has a DIFFERENT path for different people. Colab mounts a
# personal Drive and a shared Drive under separate roots, and both roots have
# a legacy spelling with a space in it. Hard-coding one path means the
# notebook works for whoever wrote it and fails for everybody else, so look
# under every root instead and report what was found.
FOLDER_NAME = 'Hmart Shared/Fixed Price Upload'

_ROOTS = ('MyDrive', 'My Drive', 'Shareddrives', 'Shared drives')

def _find_folder(name, mount='/content/drive'):
    for root in _ROOTS:
        candidate = os.path.join(mount, root, name)
        if os.path.isdir(candidate):
            return candidate
    return None

FOLDER = _find_folder(FOLDER_NAME)

if FOLDER is None:
    # Say what was looked for and what is actually there. A bare
    # FileNotFoundError two cells later sends the operator hunting.
    available = []
    for root in _ROOTS:
        base = os.path.join('/content/drive', root)
        if os.path.isdir(base):
            available += [os.path.join(root, e) for e in sorted(os.listdir(base))
                          if not e.startswith('.')]
    raise SystemExit(
        "Could not find a folder named {!r} in your Drive.\n\n"
        "Ask for access to the price folder, or open it in Drive and check "
        "the name matches exactly.\n\n"
        "What this account can see:\n  {}".format(
            FOLDER_NAME, "\n  ".join(available) or "(nothing - is Drive mounted?)"))

if not os.path.isfile(os.path.join(FOLDER, 'accounts.json')):
    raise SystemExit(
        "Found the folder:\n  {}\n\nbut it has no accounts.json, so the tool "
        "does not know which stores to write to. Ask Rossini to add it.".format(
            FOLDER))

print('Using:', FOLDER)

In [ ]:
# Cell 2 - Load your file, then check and upload.
# Building the screen and wiring its buttons happen together on purpose:
# running them as two cells let the screen be rebuilt without the buttons
# being reconnected, and the rebuilt buttons then did nothing when clicked.
# Let the screen use the full height of the output area instead of a
# short scrolling box. Colab-only, so it is allowed to be absent.
try:
    from google.colab import output as _colab_output
    _colab_output.no_vertical_scroll()
except Exception:
    pass
from vtex_fixed_price_uploader.config import load_config
from vtex_fixed_price_uploader.notebook_ui import build_ui, run_interactive
CONFIG = load_config(f'{FOLDER}/accounts.json')
UI = build_ui(CONFIG, f'{FOLDER}/write-log.jsonl', f'{FOLDER}/snapshot.json')
run_interactive(CONFIG, UI)


## Handling the login value

The login is typed into a password field and is never written to a log, a request URL, or a request body by this tool. It is **not** protected against the notebook's own autosave: Colab and Jupyter can serialise widget values into the `.ipynb` file, and this notebook lives in Drive.

So, as a handling rule:

1. Do not save the notebook while the login field still has a value.
2. Clear the login field when you finish, then save.
3. If you saved with it filled, treat that login as exposed and get a fresh one.

## If something goes wrong

- **Upload stopped part-way.** The read-back still runs and reports what landed. Get a fresh login and press *Check this file* again to finish the rest.
- **Rows unreadable.** Press *Check what landed (no writing)* later; it re-reads without touching production.
- **The wrong prices went up.** Press *Put the previous prices back*. It states what it will put back and for how many pairs, and writes nothing until you confirm.
- **A run refuses to start because an earlier one is still open.** Tick *Abandon the unfinished upload log* and check the file again. Rows already written stay written in VTEX.